# IUROPA citation edges

This notebook constructs an **observed citation-edge table** for IUROPA.

- One row = one observed citation from a citing judgment to a cited judgment.
- Node metadata are loaded from `node_features.xlsx`.
- Citations are loaded from `citations.csv`.
- Edge features are constructed from:
  - timing,
  - court / chamber / judicial personnel,
  - legal subject matter,
  - procedures,
  - parties and observers,
  - national-court referrals,
  - procedural lineage.
- Set-based similarity is kept deliberately simple for now: **shared-count + Jaccard similarity**.
- **No REM risk set is constructed in this notebook.**

Output: `edge_features.xlsx`


In [1]:
# ============================================================
# 0. CONFIGURATION
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

NODE_XLSX = Path("node_features.xlsx")
CITATIONS_CSV = Path("citations.csv")
OUTPUT_XLSX = Path("edge_features.xlsx")

if not NODE_XLSX.exists():
    raise FileNotFoundError(
        f"{NODE_XLSX} not found. Put node_features.xlsx in the same folder as this notebook."
    )

if not CITATIONS_CSV.exists():
    raise FileNotFoundError(
        f"{CITATIONS_CSV} not found. Put citations.csv in the same folder as this notebook."
    )

print("Inputs found:")
print(" ", NODE_XLSX.resolve())
print(" ", CITATIONS_CSV.resolve())


Inputs found:
  C:\Users\laze\Downloads\IUROPA\IUROPA\node_features.xlsx
  C:\Users\laze\Downloads\IUROPA\IUROPA\citations.csv


In [2]:
# ============================================================
# 1. LOAD NODES + OBSERVED CITATIONS
# ============================================================

nodes = pd.read_excel(NODE_XLSX, sheet_name="nodes")
citations = pd.read_csv(CITATIONS_CSV)

required_citation_cols = {"citing_decision_id", "cited_decision_id"}
missing = required_citation_cols - set(citations.columns)
if missing:
    raise KeyError(f"citations.csv is missing required columns: {sorted(missing)}")

if "decision_id" not in nodes.columns:
    raise KeyError("The nodes sheet must contain decision_id.")

nodes = nodes.drop_duplicates("decision_id").copy()
citations = citations.drop_duplicates(
    ["citing_decision_id", "cited_decision_id"]
).copy()

print(f"Node table:      {len(nodes):,} judgments")
print(f"Observed edges:  {len(citations):,} unique citations")

display(citations.head())


Node table:      25,778 judgments
Observed edges:  283,994 unique citations


,citing_decision_id,cited_decision_id
0,DOC_JUDG_CJ_1998_0082_P_20000525,DOC_JUDG_CJ_1995_0008_P_19980528
1,DOC_JUDG_CJ_1998_0082_P_20000525,DOC_JUDG_CJ_1983_0264_19850704
2,DOC_JUDG_CJ_2018_0025_20190508,DOC_JUDG_CJ_2017_0308_20181115
3,DOC_JUDG_CJ_2018_0025_20190508,DOC_JUDG_CJ_2016_0249_20170615
4,DOC_JUDG_CJ_2018_0025_20190508,DOC_JUDG_CJ_2016_0560_20180307


In [3]:
# ============================================================
# 2. HELPERS
# ============================================================

def to_set(x):
    """
    Convert semicolon-separated node metadata to a Python set.
    Missing values become the empty set.
    """
    if pd.isna(x):
        return set()

    if isinstance(x, (set, list, tuple)):
        vals = x
    else:
        vals = str(x).split(";")

    out = set()
    for v in vals:
        s = str(v).strip()
        if s and s.lower() not in {"nan", "none", "<na>"}:
            out.add(s)
    return out


def shared_count(a, b):
    return len(to_set(a) & to_set(b))


def jaccard(a, b):
    """
    Jaccard(A,B) = |A ∩ B| / |A ∪ B|

    Returns NaN if both sets are empty because similarity is undefined.
    Returns 0 if exactly one set is empty.
    """
    A = to_set(a)
    B = to_set(b)

    union = A | B
    if not union:
        return np.nan

    return len(A & B) / len(union)


def any_overlap(a, b):
    """1 if two non-empty sets overlap, 0 if they do not, NaN if both are empty."""
    A = to_set(a)
    B = to_set(b)

    if not A and not B:
        return np.nan

    return int(bool(A & B))


def same_scalar(a, b):
    """Exact equality for scalar metadata; NaN if either side is missing."""
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return int(str(a).strip() == str(b).strip())


def abs_diff(a, b):
    a = pd.to_numeric(pd.Series([a]), errors="coerce").iloc[0]
    b = pd.to_numeric(pd.Series([b]), errors="coerce").iloc[0]
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return abs(a - b)


def slugify(x):
    x = str(x).strip().lower()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    return x.strip("_") or "missing"


def clean_excel_string(x):
    """Remove XML control characters Excel/openpyxl cannot write."""
    if isinstance(x, str):
        x = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", x)
        if len(x) > 32767:
            x = x[:32767]
    return x


def add_set_similarity(df, citing_col, cited_col, prefix):
    """Add shared-count, overlap flag, and Jaccard similarity for two set-valued columns."""
    if citing_col not in df.columns or cited_col not in df.columns:
        return df

    df[f"n_shared_{prefix}"] = [
        shared_count(a, b) for a, b in zip(df[citing_col], df[cited_col])
    ]
    df[f"same_{prefix}"] = [
        any_overlap(a, b) for a, b in zip(df[citing_col], df[cited_col])
    ]
    df[f"{prefix}_jaccard"] = [
        jaccard(a, b) for a, b in zip(df[citing_col], df[cited_col])
    ]
    return df


In [4]:
# ============================================================
# 3. CHOOSE NODE FEATURES TO CARRY INTO THE EDGE TABLE
# ============================================================
#
# We deliberately do NOT duplicate every node column.
# We retain variables that are useful either:
#   (a) directly as sender/receiver covariates, or
#   (b) for constructing dyadic edge features.
#
# Missing columns are skipped automatically, so the notebook remains
# robust if the node notebook changes slightly.

node_feature_candidates = [
    # Core identity / timing
    "decision_id", "decision_date", "decision_year", "court",
    "case_id", "case_number", "proceeding_id", "lead_proceeding_id",
    "proceeding_number", "proceeding_name",

    # Procedure
    "procedures", "outcomes",
    "is_preliminary_ruling", "is_appeal", "is_annulment",
    "is_infringement", "is_damages_action", "is_failure_to_act",

    # Proceeding structure
    "is_joined_case", "joined_proceeding_count", "joined_proceeding_list",
    "n_proceedings_in_bundle",
    "is_appeal_of", "appeal_of_list",
    "is_transferred", "transferred_to_proceeding_id",

    # Formation / panel
    "chambers", "is_grand_chamber", "is_full_court_or_plenary",
    "panel_judge_ids", "panel_size",
    "panel_mean_age_approx", "panel_share_female",
    "panel_n_member_states", "panel_member_states",
    "panel_member_state_diversity",
    "panel_mean_cjeu_service_years",
    "panel_mean_judicial_service_years",
    "panel_mean_prior_judge_assignments",

    # Rapporteur
    "rapporteur_ids", "rapporteur_member_states",
    "rapporteur_age_approx",
    "rapporteur_cjeu_service_years",
    "rapporteur_judicial_service_years",
    "rapporteur_prior_judge_assignments",
    "rapporteur_prior_rapporteur_assignments",

    # Advocate General
    "has_advocate_general", "ag_ids", "ag_member_states",
    "ag_age_approx", "ag_cjeu_service_years",
    "ag_role_service_years", "ag_prior_assignments",

    # Legal subject matter
    "directory_codes", "directory_sections",
    "directory_major_codes", "directory_minor_codes",
    "keywords",
    "directory_code_novelty",
    "directory_section_novelty",
    "keyword_novelty",

    # Parties / observers
    "party_ids", "party_types", "party_roles", "n_parties",
    "has_commission_party",
    "observer_ids", "observer_types", "n_observers",
    "had_observers", "has_commission_observer",

    # National-court referrals
    "referring_court_ids", "referring_court_names",
    "n_referring_courts",
    "referring_member_states", "n_referring_member_states",
    "matched_referring_court_names",
    "referring_court_levels",
    "referring_court_prestiges",
    "referring_court_collegialitys",
    "referring_court_competences",
    "referring_court_competence_detaileds",
]

node_cols = [c for c in node_feature_candidates if c in nodes.columns]

print(f"Using {len(node_cols):,} node columns for edge construction.")
print("Skipped because unavailable:")
for c in node_feature_candidates:
    if c not in nodes.columns:
        print(" ", c)


Using 80 node columns for edge construction.
Skipped because unavailable:
  directory_major_codes
  directory_minor_codes


In [5]:
# ============================================================
# 4. MERGE CITING + CITED NODE FEATURES
# ============================================================

node_subset = nodes[node_cols].copy()

citing_nodes = node_subset.rename(
    columns={c: f"citing_{c}" for c in node_subset.columns if c != "decision_id"}
).rename(columns={"decision_id": "citing_decision_id"})

cited_nodes = node_subset.rename(
    columns={c: f"cited_{c}" for c in node_subset.columns if c != "decision_id"}
).rename(columns={"decision_id": "cited_decision_id"})

edges = citations.merge(
    citing_nodes,
    on="citing_decision_id",
    how="left",
    validate="many_to_one",
)

edges = edges.merge(
    cited_nodes,
    on="cited_decision_id",
    how="left",
    validate="many_to_one",
)

edges["citing_node_found"] = edges["citing_decision_date"].notna().astype(int)
edges["cited_node_found"] = edges["cited_decision_date"].notna().astype(int)
edges["both_nodes_found"] = (
    edges["citing_node_found"].eq(1) & edges["cited_node_found"].eq(1)
).astype(int)

print(f"Edges with both nodes found: {edges['both_nodes_found'].sum():,} / {len(edges):,}")
print(f"Missing citing nodes: {(edges['citing_node_found'] == 0).sum():,}")
print(f"Missing cited nodes:  {(edges['cited_node_found'] == 0).sum():,}")


Edges with both nodes found: 266,217 / 283,994
Missing citing nodes: 16,899
Missing cited nodes:  954


In [6]:
# ============================================================
# 5. BASIC TIMING FEATURES
# ============================================================

edges["citing_decision_date"] = pd.to_datetime(
    edges["citing_decision_date"], errors="coerce"
)
edges["cited_decision_date"] = pd.to_datetime(
    edges["cited_decision_date"], errors="coerce"
)

edges["citation_lag_days"] = (
    edges["citing_decision_date"] - edges["cited_decision_date"]
).dt.days

edges["citation_lag_years"] = edges["citation_lag_days"] / 365.25

# Should normally be 1: citations should point to an earlier judgment.
edges["cited_strictly_before_citing"] = np.where(
    edges["citation_lag_days"].notna(),
    (edges["citation_lag_days"] > 0).astype(int),
    np.nan,
)

edges["same_decision_year"] = [
    same_scalar(a, b)
    for a, b in zip(
        edges.get("citing_decision_year", pd.Series(index=edges.index, dtype=float)),
        edges.get("cited_decision_year", pd.Series(index=edges.index, dtype=float)),
    )
]

display(
    edges[
        [c for c in [
            "citing_decision_id", "cited_decision_id",
            "citing_decision_date", "cited_decision_date",
            "citation_lag_days", "citation_lag_years",
            "cited_strictly_before_citing"
        ] if c in edges.columns]
    ].head()
)


,citing_decision_id,cited_decision_id,citing_decision_date,cited_decision_date,citation_lag_days,citation_lag_years,cited_strictly_before_citing
0,DOC_JUDG_CJ_1998_0082_P_20000525,DOC_JUDG_CJ_1995_0008_P_19980528,2000-05-25,1998-05-28,728.0,1.993155,1.0
1,DOC_JUDG_CJ_1998_0082_P_20000525,DOC_JUDG_CJ_1983_0264_19850704,2000-05-25,1985-07-04,5439.0,14.891170,1.0
2,DOC_JUDG_CJ_2018_0025_20190508,DOC_JUDG_CJ_2017_0308_20181115,2019-05-08,2018-11-15,174.0,0.476386,1.0
3,DOC_JUDG_CJ_2018_0025_20190508,DOC_JUDG_CJ_2016_0249_20170615,2019-05-08,2017-06-15,692.0,1.894593,1.0
4,DOC_JUDG_CJ_2018_0025_20190508,DOC_JUDG_CJ_2016_0560_20180307,2019-05-08,2018-03-07,427.0,1.169062,1.0


In [7]:
# ============================================================
# 6. COURT + FORMATION + JUDICIAL-PERSONNEL EDGE FEATURES
# ============================================================

# Scalar institutional similarity
if {"citing_court", "cited_court"}.issubset(edges.columns):
    edges["same_court"] = [
        same_scalar(a, b) for a, b in zip(edges["citing_court"], edges["cited_court"])
    ]

if {"citing_panel_size", "cited_panel_size"}.issubset(edges.columns):
    edges["panel_size_abs_diff"] = [
        abs_diff(a, b) for a, b in zip(edges["citing_panel_size"], edges["cited_panel_size"])
    ]

# Set-valued overlap
set_specs = [
    ("citing_chambers", "cited_chambers", "chamber"),
    ("citing_panel_judge_ids", "cited_panel_judge_ids", "panel_judges"),
    ("citing_panel_member_states", "cited_panel_member_states", "panel_member_states"),
    ("citing_rapporteur_ids", "cited_rapporteur_ids", "rapporteur"),
    ("citing_ag_ids", "cited_ag_ids", "advocate_general"),
]

for citing_col, cited_col, prefix in set_specs:
    edges = add_set_similarity(edges, citing_col, cited_col, prefix)

# Especially interpretable personnel-continuity features.
if {"citing_rapporteur_ids", "cited_panel_judge_ids"}.issubset(edges.columns):
    edges["citing_rapporteur_sat_on_cited_panel"] = [
        any_overlap(a, b)
        for a, b in zip(edges["citing_rapporteur_ids"], edges["cited_panel_judge_ids"])
    ]

if {"cited_rapporteur_ids", "citing_panel_judge_ids"}.issubset(edges.columns):
    edges["cited_rapporteur_sits_on_citing_panel"] = [
        any_overlap(a, b)
        for a, b in zip(edges["cited_rapporteur_ids"], edges["citing_panel_judge_ids"])
    ]

# Continuous similarity in panel / individual characteristics.
numeric_pairs = [
    ("panel_mean_age_approx", "panel_mean_age_abs_diff"),
    ("panel_share_female", "panel_share_female_abs_diff"),
    ("panel_member_state_diversity", "panel_member_state_diversity_abs_diff"),
    ("panel_mean_cjeu_service_years", "panel_mean_cjeu_service_years_abs_diff"),
    ("panel_mean_prior_judge_assignments", "panel_mean_prior_judge_assignments_abs_diff"),
    ("rapporteur_age_approx", "rapporteur_age_abs_diff"),
    ("rapporteur_cjeu_service_years", "rapporteur_cjeu_service_years_abs_diff"),
    ("rapporteur_prior_judge_assignments", "rapporteur_prior_judge_assignments_abs_diff"),
    ("ag_age_approx", "ag_age_abs_diff"),
    ("ag_cjeu_service_years", "ag_cjeu_service_years_abs_diff"),
    ("ag_prior_assignments", "ag_prior_assignments_abs_diff"),
]

for base, out in numeric_pairs:
    c1 = f"citing_{base}"
    c2 = f"cited_{base}"
    if c1 in edges.columns and c2 in edges.columns:
        edges[out] = [abs_diff(a, b) for a, b in zip(edges[c1], edges[c2])]


In [8]:
# ============================================================
# 7. LEGAL-SUBJECT + PROCEDURE SIMILARITY
# ============================================================
#
# For now we keep similarity deliberately simple:
#   - number of shared items,
#   - any overlap,
#   - Jaccard index.

legal_set_specs = [
    ("citing_procedures", "cited_procedures", "procedures"),
    ("citing_outcomes", "cited_outcomes", "outcomes"),
    ("citing_directory_codes", "cited_directory_codes", "directory_codes"),
    ("citing_directory_sections", "cited_directory_sections", "directory_sections"),
    ("citing_directory_major_codes", "cited_directory_major_codes", "directory_major_codes"),
    ("citing_directory_minor_codes", "cited_directory_minor_codes", "directory_minor_codes"),
    ("citing_keywords", "cited_keywords", "keywords"),
]

for citing_col, cited_col, prefix in legal_set_specs:
    edges = add_set_similarity(edges, citing_col, cited_col, prefix)

# Highly interpretable procedure combinations.
for flag in [
    "is_preliminary_ruling",
    "is_appeal",
    "is_annulment",
    "is_infringement",
    "is_damages_action",
    "is_failure_to_act",
]:
    c1 = f"citing_{flag}"
    c2 = f"cited_{flag}"

    if c1 in edges.columns and c2 in edges.columns:
        a = pd.to_numeric(edges[c1], errors="coerce")
        b = pd.to_numeric(edges[c2], errors="coerce")
        edges[f"both_{flag.replace('is_', '')}"] = np.where(
            a.notna() & b.notna(),
            ((a == 1) & (b == 1)).astype(int),
            np.nan,
        )

# Grand-Chamber relationships.
if {"citing_is_grand_chamber", "cited_is_grand_chamber"}.issubset(edges.columns):
    a = pd.to_numeric(edges["citing_is_grand_chamber"], errors="coerce")
    b = pd.to_numeric(edges["cited_is_grand_chamber"], errors="coerce")

    edges["both_grand_chamber"] = np.where(
        a.notna() & b.notna(), ((a == 1) & (b == 1)).astype(int), np.nan
    )


In [9]:
# ============================================================
# 8. PARTIES + OBSERVERS
# ============================================================

actor_set_specs = [
    ("citing_party_ids", "cited_party_ids", "parties"),
    ("citing_party_types", "cited_party_types", "party_types"),
    ("citing_observer_ids", "cited_observer_ids", "observers"),
    ("citing_observer_types", "cited_observer_types", "observer_types"),
]

for citing_col, cited_col, prefix in actor_set_specs:
    edges = add_set_similarity(edges, citing_col, cited_col, prefix)

# Commission participation on both sides.
if {"citing_has_commission_party", "cited_has_commission_party"}.issubset(edges.columns):
    a = pd.to_numeric(edges["citing_has_commission_party"], errors="coerce")
    b = pd.to_numeric(edges["cited_has_commission_party"], errors="coerce")
    edges["commission_party_in_both"] = np.where(
        a.notna() & b.notna(), ((a == 1) & (b == 1)).astype(int), np.nan
    )

if {"citing_had_observers", "cited_had_observers"}.issubset(edges.columns):
    a = pd.to_numeric(edges["citing_had_observers"], errors="coerce")
    b = pd.to_numeric(edges["cited_had_observers"], errors="coerce")
    edges["both_had_observers"] = np.where(
        a.notna() & b.notna(), ((a == 1) & (b == 1)).astype(int), np.nan
    )


In [10]:
# ============================================================
# 9. NATIONAL-COURT REFERRAL EDGE FEATURES
# ============================================================

referral_set_specs = [
    ("citing_referring_court_ids", "cited_referring_court_ids", "referring_courts"),
    ("citing_referring_member_states", "cited_referring_member_states", "referring_member_states"),
    ("citing_referring_court_levels", "cited_referring_court_levels", "referring_court_levels"),
    ("citing_referring_court_prestiges", "cited_referring_court_prestiges", "referring_court_prestiges"),
    ("citing_referring_court_collegialitys", "cited_referring_court_collegialitys", "referring_court_collegialitys"),
    ("citing_referring_court_competences", "cited_referring_court_competences", "referring_court_competences"),
    (
        "citing_referring_court_competence_detaileds",
        "cited_referring_court_competence_detaileds",
        "referring_court_competence_detaileds",
    ),
]

for citing_col, cited_col, prefix in referral_set_specs:
    edges = add_set_similarity(edges, citing_col, cited_col, prefix)

# Convenience aliases with especially clear substantive meaning.
if "same_referring_courts" in edges.columns:
    edges["same_referring_national_court"] = edges["same_referring_courts"]

if "same_referring_member_states" in edges.columns:
    edges["same_referring_country"] = edges["same_referring_member_states"]

# Keep the actual source country/countries on both sides in the table:
# citing_referring_member_states and cited_referring_member_states.


C:\Users\laze\AppData\Local\Temp\ipykernel_28128\1280492837.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  edges["same_referring_national_court"] = edges["same_referring_courts"]
C:\Users\laze\AppData\Local\Temp\ipykernel_28128\1280492837.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  edges["same_referring_country"] = edges["same_referring_member_states"]


In [11]:
# ============================================================
# 10. PROCEDURAL-LINEAGE FEATURES
# ============================================================

if {"citing_case_id", "cited_case_id"}.issubset(edges.columns):
    edges["same_case"] = [
        same_scalar(a, b) for a, b in zip(edges["citing_case_id"], edges["cited_case_id"])
    ]

if {"citing_proceeding_id", "cited_proceeding_id"}.issubset(edges.columns):
    edges["same_proceeding"] = [
        same_scalar(a, b)
        for a, b in zip(edges["citing_proceeding_id"], edges["cited_proceeding_id"])
    ]

# Does the citing judgment belong to an appeal proceeding whose predecessor
# is the proceeding of the cited judgment?
if {"citing_appeal_of_list", "cited_proceeding_id"}.issubset(edges.columns):
    edges["citing_is_appeal_of_cited_proceeding"] = [
        int(str(cited_proc).strip() in to_set(appeal_list))
        if pd.notna(cited_proc) else np.nan
        for appeal_list, cited_proc in zip(
            edges["citing_appeal_of_list"], edges["cited_proceeding_id"]
        )
    ]

# Joined-proceeding family:
# make the current proceeding itself part of the bundle before comparing.
if {"citing_proceeding_id", "cited_proceeding_id"}.issubset(edges.columns):
    citing_joined = edges.get(
        "citing_joined_proceeding_list",
        pd.Series(np.nan, index=edges.index)
    )
    cited_joined = edges.get(
        "cited_joined_proceeding_list",
        pd.Series(np.nan, index=edges.index)
    )

    citing_bundle = [
        to_set(joined) | ({str(proc).strip()} if pd.notna(proc) else set())
        for joined, proc in zip(citing_joined, edges["citing_proceeding_id"])
    ]
    cited_bundle = [
        to_set(joined) | ({str(proc).strip()} if pd.notna(proc) else set())
        for joined, proc in zip(cited_joined, edges["cited_proceeding_id"])
    ]

    edges["same_joined_proceeding_family"] = [
        int(bool(a & b)) for a, b in zip(citing_bundle, cited_bundle)
    ]
    edges["joined_proceeding_family_jaccard"] = [
        (len(a & b) / len(a | b)) if (a | b) else np.nan
        for a, b in zip(citing_bundle, cited_bundle)
    ]


C:\Users\laze\AppData\Local\Temp\ipykernel_28128\3556675149.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  edges["same_case"] = [
C:\Users\laze\AppData\Local\Temp\ipykernel_28128\3556675149.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  edges["same_proceeding"] = [
C:\Users\laze\AppData\Local\Temp\ipykernel_28128\3556675149.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using 

In [12]:
# ============================================================
# 11. NODE-NOVELTY FEATURES CARRIED ONTO EACH EDGE
# ============================================================
#
# These are NOT new network measures. They are the node-level metadata novelty
# measures already constructed in node_features.xlsx and simply attached to
# the citing/cited endpoints.

for base in [
    "directory_code_novelty",
    "directory_section_novelty",
    "keyword_novelty",
]:
    c1 = f"citing_{base}"
    c2 = f"cited_{base}"

    if c1 in edges.columns and c2 in edges.columns:
        edges[f"{base}_abs_diff"] = [
            abs_diff(a, b) for a, b in zip(edges[c1], edges[c2])
        ]


C:\Users\laze\AppData\Local\Temp\ipykernel_28128\299456358.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  edges[f"{base}_abs_diff"] = [
C:\Users\laze\AppData\Local\Temp\ipykernel_28128\299456358.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  edges[f"{base}_abs_diff"] = [
C:\Users\laze\AppData\Local\Temp\ipykernel_28128\299456358.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once 

In [13]:
# ============================================================
# 12. SANITY CHECKS + DIAGNOSTICS
# ============================================================

# One row per unique observed citation.
if edges.duplicated(["citing_decision_id", "cited_decision_id"]).any():
    raise ValueError("Edge table contains duplicate citation pairs.")

# Flag any citation that appears contemporaneous or backwards in time.
temporal_issues = edges[
    edges["citation_lag_days"].notna() & (edges["citation_lag_days"] <= 0)
].copy()

print(f"Final observed edge table: {len(edges):,} citations x {edges.shape[1]:,} columns")
print(f"Unique citation pairs:     {edges[['citing_decision_id','cited_decision_id']].drop_duplicates().shape[0]:,}")
print(f"Both nodes found:          {edges['both_nodes_found'].sum():,}")
print(f"Temporal issues (lag <=0): {len(temporal_issues):,}")

# Missingness table
missingness = pd.DataFrame({
    "feature": edges.columns,
    "n_missing": [edges[c].isna().sum() for c in edges.columns],
    "pct_missing": [100 * edges[c].isna().mean() for c in edges.columns],
    "n_unique_nonmissing": [edges[c].nunique(dropna=True) for c in edges.columns],
}).sort_values(["pct_missing", "feature"], ascending=[False, True]).reset_index(drop=True)

display(edges.head())
display(missingness.head(40))

if len(temporal_issues):
    display(
        temporal_issues[
            [
                "citing_decision_id",
                "cited_decision_id",
                "citing_decision_date",
                "cited_decision_date",
                "citation_lag_days",
            ]
        ].head(30)
    )


Final observed edge table: 283,994 citations x 264 columns
Unique citation pairs:     283,994
Both nodes found:          266,217
Temporal issues (lag <=0): 5,965


,citing_decision_id,cited_decision_id,citing_decision_date,citing_decision_year,citing_court,citing_case_id,citing_case_number,citing_proceeding_id,citing_lead_proceeding_id,citing_proceeding_number,...,same_referring_national_court,same_referring_country,same_case,same_proceeding,citing_is_appeal_of_cited_proceeding,same_joined_proceeding_family,joined_proceeding_family_jaccard,directory_code_novelty_abs_diff,directory_section_novelty_abs_diff,keyword_novelty_abs_diff
0,DOC_JUDG_CJ_1998_0082_P_20000525,DOC_JUDG_CJ_1995_0008_P_19980528,2000-05-25,2000.0,Court of Justice,CASE_CJ_1998_0082,C-82/98,PROC_CJ_1998_0082_P,PROC_CJ_1998_0082_P,C-82/98 P,...,NaN,NaN,0.0,0.0,0.0,0,0.0,0.166667,0.0,0.0
1,DOC_JUDG_CJ_1998_0082_P_20000525,DOC_JUDG_CJ_1983_0264_19850704,2000-05-25,2000.0,Court of Justice,CASE_CJ_1998_0082,C-82/98,PROC_CJ_1998_0082_P,PROC_CJ_1998_0082_P,C-82/98 P,...,NaN,NaN,0.0,0.0,0.0,0,0.0,0.333333,0.0,0.0
2,DOC_JUDG_CJ_2018_0025_20190508,DOC_JUDG_CJ_2017_0308_20181115,2019-05-08,2019.0,Court of Justice,CASE_CJ_2018_0025,C-25/18,PROC_CJ_2018_0025,PROC_CJ_2018_0025,C-25/18,...,0.0,0.0,0.0,0.0,0.0,0,0.0,0.000000,0.0,0.0
3,DOC_JUDG_CJ_2018_0025_20190508,DOC_JUDG_CJ_2016_0249_20170615,2019-05-08,2019.0,Court of Justice,CASE_CJ_2018_0025,C-25/18,PROC_CJ_2018_0025,PROC_CJ_2018_0025,C-25/18,...,0.0,0.0,0.0,0.0,0.0,0,0.0,0.000000,0.0,0.0
4,DOC_JUDG_CJ_2018_0025_20190508,DOC_JUDG_CJ_2016_0560_20180307,2019-05-08,2019.0,Court of Justice,CASE_CJ_2018_0025,C-25/18,PROC_CJ_2018_0025,PROC_CJ_2018_0025,C-25/18,...,0.0,0.0,0.0,0.0,0.0,0,0.0,0.000000,0.0,0.0


,feature,n_missing,pct_missing,n_unique_nonmissing
0,cited_transferred_to_proceeding_id,283993,99.999648,1
1,citing_transferred_to_proceeding_id,283993,99.999648,1
2,citing_appeal_of_list,263327,92.722734,1765
3,citing_joined_proceeding_list,261630,92.125186,1563
4,cited_joined_proceeding_list,251015,88.387431,1705
5,citing_matched_referring_court_names,235180,82.811609,1302
6,citing_referring_court_collegialitys,235180,82.811609,9
7,citing_referring_court_competence_detaileds,235180,82.811609,32
8,citing_referring_court_competences,235180,82.811609,10
9,citing_referring_court_levels,235180,82.811609,10


,citing_decision_id,cited_decision_id,citing_decision_date,cited_decision_date,citation_lag_days
49,DOC_JUDG_GC_2005_0456_20100428,DOC_JUDG_GC_2005_0456_20100428,2010-04-28,2010-04-28,0.0
124,DOC_JUDG_CJ_2002_0175_20050113,DOC_JUDG_CJ_2002_0174_20050113,2005-01-13,2005-01-13,0.0
240,DOC_JUDG_CJ_2022_0639_20240905,DOC_JUDG_CJ_2022_0639_20240905,2024-09-05,2024-09-05,0.0
287,DOC_JUDG_GC_1993_0432_19950307,DOC_JUDG_GC_1993_0432_19950307,1995-03-07,1995-03-07,0.0
365,DOC_JUDG_CJ_2018_0573_20191009,DOC_JUDG_CJ_2018_0573_20191009,2019-10-09,2019-10-09,0.0
384,DOC_JUDG_CJ_1982_0010_19830714,DOC_JUDG_CJ_1982_0010_19830714,1983-07-14,1983-07-14,0.0
399,DOC_JUDG_CJ_1987_0291_19881214,DOC_JUDG_CJ_1987_0291_19881214,1988-12-14,1988-12-14,0.0
400,DOC_JUDG_CJ_2024_0212_20250508,DOC_JUDG_CJ_2024_0212_20250508,2025-05-08,2025-05-08,0.0
429,DOC_JUDG_GC_2016_0836_20190516,DOC_JUDG_GC_2016_0836_20190516,2019-05-16,2019-05-16,0.0
459,DOC_JUDG_CST_2005_0022_20061213,DOC_JUDG_CST_2005_0022_20061213,2006-12-13,2006-12-13,0.0


In [14]:
# ============================================================
# 13. FEATURE DICTIONARY
# ============================================================

feature_descriptions = {
    "citing_decision_id": "IUROPA judgment making the citation.",
    "cited_decision_id": "IUROPA judgment being cited.",
    "citation_lag_days": "Days between cited judgment and citing judgment.",
    "citation_lag_years": "citation_lag_days divided by 365.25.",
    "cited_strictly_before_citing": "1 when the cited judgment predates the citing judgment.",
    "same_court": "1 when citing and cited judgments were issued by the same CJEU court.",
    "n_shared_chamber": "Number of chamber labels shared by citing and cited judgments.",
    "chamber_jaccard": "Jaccard similarity of chamber-label sets.",
    "n_shared_panel_judges": "Number of judges sitting on both the citing and cited judgments.",
    "panel_judges_jaccard": "Jaccard similarity of the two judicial panels.",
    "same_rapporteur": "1 when the judgments share at least one Judge-Rapporteur.",
    "rapporteur_jaccard": "Jaccard similarity of rapporteur-ID sets.",
    "same_advocate_general": "1 when the judgments share at least one Advocate General.",
    "advocate_general_jaccard": "Jaccard similarity of AG-ID sets.",
    "citing_rapporteur_sat_on_cited_panel": "1 when a rapporteur of the citing judgment sat on the cited judgment's panel.",
    "cited_rapporteur_sits_on_citing_panel": "1 when a rapporteur of the cited judgment sits on the citing judgment's panel.",
    "n_shared_directory_codes": "Number of exact EUR-Lex directory codes shared by the two judgments.",
    "directory_codes_jaccard": "Jaccard similarity of exact EUR-Lex directory-code sets.",
    "directory_sections_jaccard": "Jaccard similarity of top-level directory-section sets.",
    "directory_major_codes_jaccard": "Jaccard similarity of major directory-code sets.",
    "directory_minor_codes_jaccard": "Jaccard similarity of minor directory-code sets.",
    "keywords_jaccard": "Jaccard similarity of EUR-Lex keyword sets.",
    "procedures_jaccard": "Jaccard similarity of legal-procedure sets.",
    "parties_jaccard": "Jaccard similarity of standardized party-ID sets.",
    "observers_jaccard": "Jaccard similarity of observer-ID sets.",
    "same_referring_national_court": "1 when preliminary-ruling judgments share at least one referring national court.",
    "same_referring_country": "1 when preliminary-ruling judgments share at least one referring Member State.",
    "referring_courts_jaccard": "Jaccard similarity of referring-national-court ID sets.",
    "referring_member_states_jaccard": "Jaccard similarity of referring Member-State sets.",
    "same_case": "1 when citing and cited judgments belong to the same IUROPA case.",
    "same_proceeding": "1 when citing and cited judgments belong to the same IUROPA proceeding.",
    "citing_is_appeal_of_cited_proceeding": "1 when the citing judgment's appeal-of list contains the cited judgment's proceeding.",
    "same_joined_proceeding_family": "1 when the two judgments' proceeding bundles overlap.",
    "joined_proceeding_family_jaccard": "Jaccard similarity of the two proceeding bundles.",
}

feature_dictionary = pd.DataFrame({
    "feature": edges.columns,
    "description": [
        feature_descriptions.get(
            c,
            (
                "Citing-node attribute copied from node_features.xlsx."
                if c.startswith("citing_")
                else "Cited-node attribute copied from node_features.xlsx."
                if c.startswith("cited_")
                else "Constructed citation-edge feature."
            ),
        )
        for c in edges.columns
    ],
})

display(feature_dictionary.head(50))


,feature,description
0,citing_decision_id,IUROPA judgment making the citation.
1,cited_decision_id,IUROPA judgment being cited.
2,citing_decision_date,Citing-node attribute copied from node_feature...
3,citing_decision_year,Citing-node attribute copied from node_feature...
4,citing_court,Citing-node attribute copied from node_feature...
5,citing_case_id,Citing-node attribute copied from node_feature...
6,citing_case_number,Citing-node attribute copied from node_feature...
7,citing_proceeding_id,Citing-node attribute copied from node_feature...
8,citing_lead_proceeding_id,Citing-node attribute copied from node_feature...
9,citing_proceeding_number,Citing-node attribute copied from node_feature...


In [ ]:
# ============================================================
# 14. EXCEL CLEANUP + EXPORT
# ============================================================

# Remove Excel-incompatible control characters.
for df in [edges, feature_dictionary, missingness]:
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(clean_excel_string)

# Put the two edge IDs first, followed by useful timing columns.
front = [
    c for c in [
        "citing_decision_id",
        "cited_decision_id",
        "citing_decision_date",
        "cited_decision_date",
        "citation_lag_days",
        "citation_lag_years",
        "cited_strictly_before_citing",
        "both_nodes_found",
    ] if c in edges.columns
]
rest = [c for c in edges.columns if c not in front]
edges = edges[front + rest]

with pd.ExcelWriter(
    OUTPUT_XLSX,
    engine="openpyxl",
    datetime_format="yyyy-mm-dd",
    date_format="yyyy-mm-dd",
) as writer:
    edges.to_excel(writer, sheet_name="edges", index=False)
    feature_dictionary.to_excel(writer, sheet_name="feature_dictionary", index=False)
    missingness.to_excel(writer, sheet_name="missingness", index=False)

print(f"Saved: {OUTPUT_XLSX.resolve()}")
print(f"Rows: {len(edges):,}")
print(f"Columns: {edges.shape[1]:,}")
